# Random Forest para fraude con tarjetas de credito

Este notebook genera un dataset sintetico de transacciones con tarjetas, realiza EDA y entrena modelos Random Forest con diferentes configuraciones, incluyendo ajuste de hiperparametros y analisis OOB.

## 1. Librerias

In [1]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 2. Generacion de datos sinteticos

In [2]:

n = 5000
amount = np.random.lognormal(mean=3.3, sigma=0.9, size=n)
avg_amount_1m = np.random.lognormal(mean=3.1, sigma=0.7, size=n)
card_age_months = np.random.exponential(scale=24.0, size=n)

is_international = np.random.binomial(1, 0.2, size=n)
is_night = np.random.binomial(1, 0.3, size=n)

logit = 0.8*(amount>(avg_amount_1m*2)) + 1.1*is_international + 0.7*is_night - 0.002*card_age_months
logit += np.random.normal(0,0.5,size=n)
prob_fraud = 1/(1+np.exp(-logit))
y = (np.random.rand(n)<prob_fraud*0.3).astype(int)

df = pd.DataFrame({
    "amount":amount,
    "avg_amount_1m":avg_amount_1m,
    "card_age_months":card_age_months,
    "is_international":is_international,
    "is_night":is_night,
    "is_fraud":y
})
df.head()


,amount,avg_amount_1m,card_age_months,is_international,is_night,is_fraud
0,42.395522,16.500129,9.300241,0,0,0
1,23.940274,16.161149,2.609300,0,0,0
2,48.565805,6.315771,1.788388,0,1,0
3,106.775326,17.618278,2.376495,0,1,0
4,21.960864,37.076389,20.984540,0,0,0


## 3. EDA rapido

In [3]:
df.describe()

,amount,avg_amount_1m,card_age_months,is_international,is_night,is_fraud
count,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000
mean,40.758768,28.330253,24.229876,0.208400,0.292800,0.190600
std,46.512225,22.762191,24.201064,0.406205,0.455093,0.392814
min,1.466472,1.425211,0.006060,0.000000,0.000000,0.000000
25%,14.997529,13.727374,6.936721,0.000000,0.000000,0.000000
50%,27.443219,21.928446,16.920260,0.000000,0.000000,0.000000
75%,49.373288,35.661322,33.730948,0.000000,1.000000,0.000000
max,928.540442,262.522983,227.897189,1.000000,1.000000,1.000000


In [4]:

# Correlacion
df.corr(numeric_only=True)


,amount,avg_amount_1m,card_age_months,is_international,is_night,is_fraud
amount,1.000000,-0.012660,-0.002216,-0.005327,-0.009275,0.026194
avg_amount_1m,-0.012660,1.000000,-0.013878,-0.020659,-0.003792,-0.015427
card_age_months,-0.002216,-0.013878,1.000000,0.007884,-0.006074,-0.030844
is_international,-0.005327,-0.020659,0.007884,1.000000,-0.015255,0.108311
is_night,-0.009275,-0.003792,-0.006074,-0.015255,1.000000,0.071573
is_fraud,0.026194,-0.015427,-0.030844,0.108311,0.071573,1.000000


## 4. Preparacion de datos

In [5]:

X = df.drop(columns=["is_fraud"])
y = df["is_fraud"]
num_cols = X.columns.tolist()

numeric_transformer = Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",MinMaxScaler())])
preprocess = ColumnTransformer([("num",numeric_transformer,num_cols)])

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.25,stratify=y,random_state=RANDOM_STATE)


## 5. Random Forest baseline

In [6]:

rf_baseline = Pipeline([("prep",preprocess),
                        ("rf",RandomForestClassifier(n_estimators=200,random_state=RANDOM_STATE,oob_score=True))])
rf_baseline.fit(X_train,y_train)
y_pred = rf_baseline.predict(X_test)
print(classification_report(y_test,y_pred))
print("AUC:",roc_auc_score(y_test,rf_baseline.predict_proba(X_test)[:,1]))
print("OOB accuracy:",rf_baseline.named_steps["rf"].oob_score_)


              precision    recall  f1-score   support

           0       0.81      0.97      0.89      1012
           1       0.26      0.04      0.07       238

    accuracy                           0.80      1250
   macro avg       0.53      0.51      0.48      1250
weighted avg       0.71      0.80      0.73      1250

AUC: 0.533360182017471
OOB accuracy: 0.7954666666666667


## 6. Random Forest con entropia y profundidad

In [7]:

rf_entropy = Pipeline([("prep",preprocess),
                       ("rf",RandomForestClassifier(n_estimators=400,max_depth=12,criterion="entropy",random_state=RANDOM_STATE))])
rf_entropy.fit(X_train,y_train)
print(classification_report(y_test,rf_entropy.predict(X_test)))


              precision    recall  f1-score   support

           0       0.81      0.99      0.89      1012
           1       0.33      0.01      0.02       238

    accuracy                           0.81      1250
   macro avg       0.57      0.50      0.46      1250
weighted avg       0.72      0.81      0.73      1250



## 7. Busqueda de hiperparametros

In [8]:

param_distributions = {
    "rf__n_estimators":[200,400,600],
    "rf__max_depth":[None,10,12,20],
    "rf__criterion":["gini","entropy"]
}
rf_pipe = Pipeline([("prep",preprocess),("rf",RandomForestClassifier(random_state=RANDOM_STATE))])
search = RandomizedSearchCV(rf_pipe,param_distributions,n_iter=5,scoring="roc_auc",cv=3,random_state=RANDOM_STATE)
search.fit(X_train,y_train)
search.best_params_


{'rf__n_estimators': 400, 'rf__max_depth': 10, 'rf__criterion': 'entropy'}

## 8. Modelo final con mejores hiperparametros

In [9]:

best_model = search.best_estimator_
print(classification_report(y_test,best_model.predict(X_test)))


              precision    recall  f1-score   support

           0       0.81      1.00      0.89      1012
           1       0.50      0.01      0.02       238

    accuracy                           0.81      1250
   macro avg       0.66      0.50      0.46      1250
weighted avg       0.75      0.81      0.73      1250

